In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import clip
from torch.nn import functional as F
import torch.nn as nn
from torchvision import transforms
from PIL import Image

train = False
classes = None
pictures = None


def load_data():
    data_list = []
    label_list = []
    texts = []
    images = []

    if train:
        text_directory = "/home/diaoyueqin/hcy/images_set/training_images"
    else:
        text_directory = "/home/diaoyueqin/hcy/images_set/test_images"

    dirnames = [d for d in os.listdir(text_directory) if os.path.isdir(os.path.join(text_directory, d))]
    dirnames.sort()

    if classes is not None:
        dirnames = [dirnames[i] for i in classes]

    for dir in dirnames:

        try:
            idx = dir.index('_')
            description = dir[idx + 1:]
        except ValueError:
            print(f"Skipped: {dir} due to no '_' found.")
            continue

        new_description = f"{description}"
        texts.append(new_description)

    if train:
        img_directory = "/home/diaoyueqin/hcy/images_set/training_images"
    else:
        img_directory = "/home/diaoyueqin/hcy/images_set/test_images"

    all_folders = [d for d in os.listdir(img_directory) if os.path.isdir(os.path.join(img_directory, d))]
    all_folders.sort()

    if classes is not None and pictures is not None:
        images = []
        for i in range(len(classes)):
            class_idx = classes[i]
            pic_idx = pictures[i]
            if class_idx < len(all_folders):
                folder = all_folders[class_idx]
                folder_path = os.path.join(img_directory, folder)
                all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                all_images.sort()
                if pic_idx < len(all_images):
                    images.append(os.path.join(folder_path, all_images[pic_idx]))
    elif classes is not None and pictures is None:
        images = []
        for i in range(len(classes)):
            class_idx = classes[i]
            if class_idx < len(all_folders):
                folder = all_folders[class_idx]
                folder_path = os.path.join(img_directory, folder)
                all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                all_images.sort()
                images.extend(os.path.join(folder_path, img) for img in all_images)
    elif classes is None:
        images = []
        for folder in all_folders:
            folder_path = os.path.join(img_directory, folder)
            all_images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
            all_images.sort()
            images.extend(os.path.join(folder_path, img) for img in all_images)
    else:

        print("Error")
    return texts, images


# texts, images = load_data()
# images

/home/diaoyueqin/anaconda3/envs/BCI/lib/python3.10/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


In [ ]:
# texts

In [ ]:
import os

import torch
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader

os.environ["WANDB_API_KEY"] = "KEY"
os.environ["WANDB_MODE"] = 'offline'
from itertools import combinations

import clip
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
import tqdm
from eegdatasets_leaveone import EEGDataset

from einops.layers.torch import Rearrange, Reduce

from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset
import random
from util import wandb_logger
from braindecode.models import EEGNetv4, ATCNet, EEGConformer, EEGITNet, ShallowFBCSPNet
import csv
from torch import Tensor
import itertools
import math
import re
from subject_layers.Transformer_EncDec import Encoder, EncoderLayer
from subject_layers.SelfAttention_Family import FullAttention, AttentionLayer
from subject_layers.Embed import DataEmbedding
from subject_layers.mamba2 import Mamba2Encoder
from subject_layers.MontageAwareEmbedding import MontageAwareEmbedding
import numpy as np
from loss import ClipLoss
import argparse
from torch import nn
from torch.optim import AdamW

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model + 1, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term[:d_model // 2 + 1])
        pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])

        self.register_buffer('pe', pe)

    def forward(self, x):
        pe = self.pe[:x.size(0), :].unsqueeze(1).repeat(1, x.size(1), 1)
        x = x + pe
        return x

def build_coords_from_ch_names(ch_names, montage_name="standard_1020"):
    """
    ch_names: list[str]，来自你的预处理产物
    return: torch.FloatTensor [C,3]
    """
    try:
        import mne
    except Exception as e:
        raise RuntimeError("mne 未安装或不可用，但 montage-aware 需要它来生成标准电极坐标") from e

    montage = mne.channels.make_standard_montage(montage_name)
    ch_pos = montage.get_positions()["ch_pos"]  # dict: name -> (x,y,z)

    # 做一个大小写/前缀鲁棒的映射
    def norm(n):
        n = n.strip()
        n = n.replace("EEG ", "").replace("eeg ", "")
        return n

    ch_pos_lut = {k.lower(): v for k, v in ch_pos.items()}

    coords = []
    missing = []
    for name in ch_names:
        key = norm(name)
        v = ch_pos_lut.get(key.lower(), None)
        if v is None:
            # 尝试常见大小写形式（Fp1 vs FP1）
            v = ch_pos_lut.get(key.capitalize().lower(), None)
        if v is None:
            missing.append(name)
            v = (0.0, 0.0, 0.0)  # 找不到就置零，不让程序崩
        coords.append(v)

    if len(missing) > 0:
        print(f"[MontageAware] WARNING: {len(missing)} channels not found in {montage_name}, set to zeros. "
              f"Examples: {missing[:5]}")

    coords = torch.tensor(coords, dtype=torch.float32)  # [C,3]
    return coords


class Config:
    def __init__(self):
        self.task_name = 'classification'  # Example task name
        self.seq_len = 250  # Sequence length
        self.pred_len = 250  # Prediction length
        self.output_attention = False  # Whether to output attention weights
        self.d_model = 250  # Model dimension
        self.embed = 'timeF'  # Time encoding method
        self.freq = 'h'  # Time frequency
        self.dropout = 0.25  # Dropout rate
        self.factor = 1  # Attention scaling factor
        self.n_heads = 4  # Number of attention heads
        self.e_layers = 1  # Number of encoder layers
        self.d_ff = 256  # Dimension of the feedforward network
        self.activation = 'gelu'  # Activation function
        self.enc_in = 63  # Encoder input dimension (example value)


class iTransformer(nn.Module):
    def __init__(self, configs, joint_train=False, num_subjects=10, backbone="mamba2"):
        super(iTransformer, self).__init__()
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.output_attention = configs.output_attention
        # Embedding
        #self.enc_embedding = DataEmbedding(configs.seq_len, configs.d_model, configs.embed, configs.freq,
        #                                   configs.dropout, joint_train=False, num_subjects=num_subjects)
        self.enc_embedding = MontageAwareEmbedding(
            configs.seq_len, configs.d_model, configs.embed, configs.freq, configs.dropout,
            joint_train=False, num_subjects=num_subjects,
            coord_dim=3, coord_scale=1.0, use_coords=True
        )
        self.pos_encoder = PositionalEncoding(configs.enc_in)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=configs.enc_in, nhead=1)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)

        # Encoder

        if backbone == "mamba2":
            self.encoder = Mamba2Encoder(
                d_model=configs.d_model,      # 250
                n_layers=4,    # 与 baseline 对齐 configs.e_layers 为 4 改为 2 又改回
                dropout=0.2,    # 与 baseline 对齐 configs.dropout 为 0.25 改为 0.2
                d_state=128,
                d_conv=4,
                expand=1,  # 2 改 1
                headdim=25,                  # 50 改 25
                use_mem_eff_path=True,
                chunk_size=128,      # 256 改 128
            )
            print("configs.d_model =", configs.d_model)

        else:
            self.encoder = Encoder(
                [
                    EncoderLayer(
                        AttentionLayer(
                            FullAttention(False, configs.factor, attention_dropout=configs.dropout,
                                          output_attention=configs.output_attention),
                            configs.d_model, configs.n_heads
                        ),
                        configs.d_model,
                        configs.d_ff,
                        dropout=configs.dropout,
                        activation=configs.activation
                    ) for l in range(configs.e_layers)
                ],
                norm_layer=torch.nn.LayerNorm(configs.d_model)
            )

    def forward(self, x_enc, x_mark_enc, subject_ids=None):
        #embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc, subject_ids)  # [B,64,250]
        enc_out, _ = self.encoder(enc_out, attn_mask=None)            # [B,64,250]
        # enc_out = enc_out[:, :63, :]                                  # [B,63,250]
        # return enc_out
        # 如果 embedding 里拼了 subject token，就去掉它
        if getattr(self.enc_embedding, "subject_embedding", None) is not None:
            enc_out = enc_out[:, 1:, :]  # 或者 enc_out = enc_out[:, -63:, :]

        # 建议加个断言防止以后又切错
        assert enc_out.size(1) == 63
        return enc_out
        # print("enc_out", enc_out.shape)

class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        # Revised from ShallowNet
        self.tsconv = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), stride=(1, 1)),
            nn.AvgPool2d((1, 51), (1, 5)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Conv2d(40, 40, (63, 1), stride=(1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Dropout(0.5),
        )

        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e (h) (w) -> b (h w) e'),
        )

    def forward(self, x: Tensor) -> Tensor:
        # b, _, _, _ = x.shape
        x = x.unsqueeze(1)
        # print("x", x.shape)   
        x = self.tsconv(x)
        # print("tsconv", x.shape)   
        x = self.projection(x)
        # print("projection", x.shape)  
        return x


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FlattenHead(nn.Sequential):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        return x


class Enc_eeg(nn.Sequential):
    def __init__(self, emb_size=40, **kwargs):
        super().__init__(
            PatchEmbedding(emb_size),
            FlattenHead()
        )


class Proj_eeg(nn.Sequential):
    def __init__(self, embedding_dim=1440, proj_dim=1024, drop_proj=0.5):
        super().__init__(
            nn.Linear(embedding_dim, proj_dim),
            ResidualAdd(nn.Sequential(
                nn.GELU(),
                nn.Linear(proj_dim, proj_dim),
                nn.Dropout(drop_proj),
            )),
            nn.LayerNorm(proj_dim),
        )


class ATMS(nn.Module):
    def __init__(self, num_channels=63, sequence_length=25, num_subjects=1, num_features=64, num_latents=1024,
                 num_blocks=1):
        super(ATMS, self).__init__()
        default_config = Config()
        self.encoder = iTransformer(default_config, backbone="mamba2")
        self.subject_wise_linear = nn.ModuleList(
            [nn.Linear(default_config.d_model, sequence_length) for _ in range(num_subjects)])
        self.enc_eeg = Enc_eeg()
        self.proj_eeg = Proj_eeg()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.loss_func = ClipLoss()

    def forward(self, x, subject_ids):
        x = self.encoder(x, None, subject_ids)
        # print(f'After attention shape: {x.shape}')
        # print("x", x.shape)
        # x = self.subject_wise_linear[0](x)
        # print(f'After subject-specific linear transformation shape: {x.shape}')
        eeg_embedding = self.enc_eeg(x)

        out = self.proj_eeg(eeg_embedding)
        return out


def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None


def get_eegfeatures(sub, eegmodel, dataloader, device, text_features_all, img_features_all, k):
    eegmodel.eval()
    text_features_all = text_features_all.to(device).float()
    img_features_all = img_features_all.to(device).float()
    total_loss = 0
    correct = 0
    total = 0
    alpha = 0.9
    top5_correct = 0
    top5_correct_count = 0

    all_labels = set(range(text_features_all.size(0)))
    top5_acc = 0
    mse_loss_fn = nn.MSELoss()
    ridge_lambda = 0.1
    save_features = True
    features_list = []  # List to store features    
    with torch.no_grad():
        for batch_idx, (eeg_data, labels, text, text_features, img, img_features) in enumerate(dataloader):
            eeg_data = eeg_data.to(device)
            text_features = text_features.to(device).float()
            labels = labels.to(device)
            img_features = img_features.to(device).float()

            batch_size = eeg_data.size(0)  # Assume the first element is the data tensor
            subject_id = extract_id_from_string(sub)
            # eeg_data = eeg_data.permute(0, 2, 1)
            subject_ids = torch.full((batch_size,), subject_id, dtype=torch.long).to(device)
            # if not config.insubject:
            #     subject_ids = torch.full((batch_size,), -1, dtype=torch.long).to(device)          
            eeg_features = eeg_model(eeg_data, subject_ids)
            features_list.append(eeg_features)

            logit_scale = eeg_model.logit_scale

            regress_loss = mse_loss_fn(eeg_features, img_features)
            # print("eeg_features", eeg_features.shape)
            # print(torch.std(eeg_features, dim=-1))
            # print(torch.std(img_features, dim=-1))
            # l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
            # loss = (regress_loss + ridge_lambda * l2_norm)
            img_loss = eegmodel.loss_func(eeg_features, img_features, logit_scale)
            text_loss = eegmodel.loss_func(eeg_features, text_features, logit_scale)
            contrastive_loss = img_loss
            # loss = img_loss + text_loss

            regress_loss =  mse_loss_fn(eeg_features, img_features)
            # print("text_loss", text_loss)
            # print("img_loss", img_loss)
            # print("regress_loss", regress_loss)            
            # l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
            # loss = (regress_loss + ridge_lambda * l2_norm)       
            loss = alpha * regress_loss *10 + (1 - alpha) * contrastive_loss*10

            # print("loss", loss)
            total_loss += loss.item()

            for idx, label in enumerate(labels):

                possible_classes = list(all_labels - {label.item()})
                selected_classes = random.sample(possible_classes, k - 1) + [label.item()]
                selected_img_features = img_features_all[selected_classes]

                logits_img = logit_scale * eeg_features[idx] @ selected_img_features.T
                # logits_text = logit_scale * eeg_features[idx] @ selected_text_features.T
                # logits_single = (logits_text + logits_img) / 2.0
                logits_single = logits_img
                # print("logits_single", logits_single.shape)

                # predicted_label = selected_classes[torch.argmax(logits_single).item()]
                predicted_label = selected_classes[
                    torch.argmax(logits_single).item()]  # (n_batch, ) \in {0, 1, ..., n_cls-1}
                if predicted_label == label.item():
                    correct += 1
                total += 1

        if save_features:
            print(f"features_list length: {len(features_list)}")
            features_tensor = torch.cat(features_list, dim=0)
            print("features_tensor", features_tensor.shape)
            torch.save(features_tensor.cpu(),
                       f"/home/diaoyueqin/hcy/Generation/ATM_S_mamto2_eeg_features_{sub}.pt")  # Save features as .pt file
    average_loss = total_loss / (batch_idx + 1)
    accuracy = correct / total
    return average_loss, accuracy, labels, features_tensor.cpu()


from IPython.display import Image, display

config = {
    "data_path": "/home/diaoyueqin/hcy/Preprocessed_data_250Hz",
    "project": "train_pos_img_text_rep",
    "entity": "sustech_rethinkingbci",
    "name": "lr=3e-4_img_pos_pro_eeg",
    "lr": 3e-4,
    "epochs": 50,
    "batch_size": 1024,
    "logger": True,
    "encoder_type": 'ATMS',
}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data_path = config['data_path']
emb_img_test = torch.load('/home/diaoyueqin/hcy/CLIP/ViT-H-14_features_test.pt')
emb_img_train = torch.load('/home/diaoyueqin/hcy/CLIP/ViT-H-14_features_train.pt')

eeg_model = ATMS(63, 250)

print('number of parameters:', sum([p.numel() for p in eeg_model.parameters()]))

#####################################################################################

# eeg_model.load_state_dict(torch.load("/home/ldy/Workspace/Reconstruction/models/contrast/sub-08/01-30_00-44/40.pth"))
#eeg_model.load_state_dict(torch.load("/home/diaoyueqin/hcy/Generation/models/contrast/ATMS/sub-08/12-23_21-12/40.pth"))
# state_dict = torch.load("/home/diaoyueqin/hcy/Generation/models/contrast/ATMS/sub-08/01-27_22-41/20.pth")
# state_dict = torch.load("/home/diaoyueqin/hcy/Generation/models/contrast/ATMS/mamba_token/sub-08/01-28_15-41/35.pth") # test accuracy 0.35 train 0.0045
state_dict = torch.load("/home/diaoyueqin/hcy/Generation/models/contrast/ATMS/mamba_token/sub-08/01-28_22-12/20.pth") # test accuracy 0.36 train 0.0047
del state_dict['subject_wise_linear.1.weight']
del state_dict['subject_wise_linear.1.bias']
eeg_model.load_state_dict(state_dict)
eeg_model = eeg_model.to(device)
sub = 'sub-08'

# #####################################################################################
# train_dataset = EEGDataset(data_path, subjects= [sub], train=True)
# # dataset 已经创建完成后：
# ch_names = train_dataset.ch_names  # 你 eegdatasets_leaveone.py 里确实保存了 self.ch_names
# 
# coords = build_coords_from_ch_names(ch_names, montage_name="standard_1020").to(device)
# 
# # 把 coords 注入到模型的 embedding 前端
# # 你需要保证你的 enc_embedding 有一个 set_coords() 方法
# eeg_model.encoder.enc_embedding.set_coords(coords)
# print("[MontageAware] coords injected into encoder embedding:", coords.shape)
# test_dataset = EEGDataset(data_path, subjects=[sub], train=False)
# test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
# text_features_test_all = test_dataset.text_features
# img_features_test_all = test_dataset.img_features
# test_loss, test_accuracy,labels, eeg_features_test = get_eegfeatures(sub, eeg_model, test_loader, device, text_features_test_all, img_features_test_all,k=200)
# print(f" - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

In [9]:
#####################################################################################
train_dataset = EEGDataset(data_path, subjects= [sub], train=True)
# dataset 已经创建完成后：
ch_names = train_dataset.ch_names  # 你 eegdatasets_leaveone.py 里确实保存了 self.ch_names

coords = build_coords_from_ch_names(ch_names, montage_name="standard_1020").to(device)

# 把 coords 注入到模型的 embedding 前端
# 你需要保证你的 enc_embedding 有一个 set_coords() 方法
eeg_model.encoder.enc_embedding.set_coords(coords)
print("[MontageAware] coords injected into encoder embedding:", coords.shape)
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
text_features_train_all = train_dataset.text_features
img_features_train_all = train_dataset.img_features

train_loss, train_accuracy, labels, eeg_features_train = get_eegfeatures(sub, eeg_model, train_loader, device, text_features_train_all, img_features_train_all,k=200)
print(f" - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
#####################################################################################

self.subjects ['sub-08']
exclude_subject None
data_tensor torch.Size([66160, 63, 250])
Data tensor shape: torch.Size([66160, 63, 250]), label tensor shape: torch.Size([66160]), text length: 1654, image length: 16540
[MontageAware] coords injected into encoder embedding: torch.Size([63, 3])
features_list length: 65
features_tensor torch.Size([66160, 1024])
 - Train Loss: 3.4233, Train Accuracy: 0.0047


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import open_clip
from matplotlib.font_manager import FontProperties

import sys
from diffusion_prior import *
from custom_pipeline_CA_new import *
from rectified_flow_prior import RFPipe
# os.environ["CUDA_VISIBLE_DEVICES"] = "5" 
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [3]:
emb_img_test = torch.load('/home/diaoyueqin/hcy/CLIP/ViT-H-14_features_test.pt')
emb_img_train = torch.load('/home/diaoyueqin/hcy/CLIP/ViT-H-14_features_train.pt')
emb_img_train_feats = emb_img_train["img_features"]  # 取出图像特征
emb_text_train_feats = emb_img_train["text_features"]  # 如果后面也要用文字特征，可顺便取出
print("img_feats shape:", emb_img_train_feats.shape)
print("text_feats shape:", emb_text_train_feats.shape)
emb_img_train_4 = emb_img_train_feats.view(1654, 10, 1, 1024).repeat(1, 1, 4, 1).view(-1, 1024)
emb_eeg = torch.load('/home/diaoyueqin/hcy/Generation/ATM_S_mamto4_eeg_features_sub-08.pt')
emb_eeg_test = torch.load('/home/diaoyueqin/hcy/Generation/ATM_S_mamto4_eeg_features_sub-08_test.pt')
print("eeg_features_train:", emb_eeg.shape)     # 你 dataset 用的是它
print("emb_img_train_4:", emb_img_train_4.shape)

img_feats shape: torch.Size([16540, 1024])
text_feats shape: torch.Size([1654, 1024])
eeg_features_train: torch.Size([66160, 1024])
emb_img_train_4: torch.Size([66160, 1024])


In [4]:
emb_eeg.shape, emb_eeg_test.shape

(torch.Size([66160, 1024]), torch.Size([200, 1024]))

In [4]:
emb_eeg

tensor([[-0.1853,  0.0595,  0.0798,  ...,  0.0615, -0.1045,  0.1346],
        [-0.1745,  0.3108,  0.0619,  ...,  0.0423, -0.0224,  0.0317],
        [ 0.0990,  0.0182, -0.0222,  ..., -0.0843,  0.0459,  0.0090],
        ...,
        [ 0.0837,  0.0782,  0.1116,  ...,  0.0718, -0.1616,  0.1876],
        [ 0.0459,  0.0775,  0.0584,  ...,  0.0401, -0.0315, -0.0404],
        [-0.0647,  0.0769, -0.1030,  ..., -0.0678, -0.1218,  0.1716]])

In [ ]:
dataset = EmbeddingDataset(
    c_embeddings=emb_eeg, h_embeddings=emb_img_train_4, 
    # h_embeds_uncond=h_embeds_imgnet
)
dl = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=64)
diffusion_prior = DiffusionPriorUNet(cond_dim=1024, dropout=0.1)
# number of parameters
print(sum(p.numel() for p in diffusion_prior.parameters() if p.requires_grad))
pipe = RFPipe(diffusion_prior, device=device)

# load pretrained model
model_name = 'diffusion_prior' # 'diffusion_prior_vice_pre_imagenet' or 'diffusion_prior_vice_pre'
# pipe.train(dl, num_epochs=150, learning_rate=1e-3) #0.0362
# pipe.diffusion_prior.load_state_dict(torch.load(f'/home/diaoyueqin/hcy/Generation/fintune_ckpts/{config["encoder_type"]}/{sub}/{model_name}_RFlow_mamto2_150_e-3.pt', map_location=device))

In [6]:
# # pipe.diffusion_prior.load_state_dict(torch.load(f'/home/diaoyueqin/hcy/Generation/fintune_ckpts/{config['data_path']}/{sub}/{model_name}_RFlow1.pt', map_location=device))
config = {
    "data_path": "/home/diaoyueqin/hcy/Preprocessed_data_250Hz",
    "project": "train_pos_img_text_rep",
    "entity": "sustech_rethinkingbci",
    "name": "lr=3e-4_img_pos_pro_eeg",
    "lr": 3e-4,
    "epochs": 50,
    "batch_size": 1024,
    "logger": True,
    "encoder_type": 'ATMS',
}
sub = "sub-08"
# save_path = f'/home/diaoyueqin/hcy/Generation/fintune_ckpts/{config["encoder_type"]}/{sub}/{model_name}_RFlow_mamto2_150_e-3.pt'
# 
# directory = os.path.dirname(save_path)
# 
# # Create the directory if it doesn't exist
# os.makedirs(directory, exist_ok=True)
# torch.save(pipe.diffusion_prior.state_dict(), save_path)
pipe.diffusion_prior.load_state_dict(torch.load(f'/home/diaoyueqin/hcy/Generation/fintune_ckpts/{config["encoder_type"]}/{sub}/{model_name}_RFlow_mamto4.pt', map_location=device))

<All keys matched successfully>

In [16]:
# emb_img_test = torch.load('/home/diaoyueqin/hcy/CLIP/ViT-H-14_features_test.pt')
# emb_img_test_feats = emb_img_test["img_features"]
# print("img_feats_test shape:", emb_img_test_feats.shape)
# print("emb_eeg_test shape:", emb_img_test_feats.shape)
# # 你按 k in range(200) 生成，所以这里也取 200 个作为 GT
# img_test_200 = emb_img_test_feats[:200]
# eeg_test_200 = emb_eeg_test[:200]

img_feats_test shape: torch.Size([200, 1024])
emb_eeg_test shape: torch.Size([200, 1024])


In [ ]:
# import torch
# import torch.nn.functional as F
# 
# @torch.no_grad()
# def prior_only_eval(pipe, eeg_test_200, img_test_200, steps, guidance=5.0, device="cuda"):
#     pipe.diffusion_prior.eval()
#     eeg = eeg_test_200.to(device).float()
#     gt  = img_test_200.to(device).float()
# 
#     gtn = F.normalize(gt, dim=-1)
# 
#     zhat = pipe.generate(c_embeds=eeg, num_inference_steps=steps, guidance_scale=guidance)
#     zhatn = F.normalize(zhat, dim=-1)
# 
#     cos = (zhatn * gtn).sum(-1)                 # [200]
#     sims = zhatn @ gtn.T                        # [200,200]
#     top5 = sims.topk(k=5, dim=-1).indices       # [200,5]
#     idx  = torch.arange(gtn.shape[0], device=device).view(-1,1)
# 
#     top1 = (top5[:, :1] == idx).float().mean().item()
#     top5acc = (top5 == idx).any(dim=-1).float().mean().item()
# 
#     return float(cos.mean().item()), float(cos.std(unbiased=False).item()), top1, top5acc
# 
# for s in [50, 32, 16, 8]:
#     cos_m, cos_std, top1, top5 = prior_only_eval(pipe, eeg_test_200, img_test_200, steps=s, guidance=5.0, device=device)
#     print(f"[RFlowmamto] steps={s} cos={cos_m:.4f}±{cos_std:.4f} top1={top1:.3f} top5={top5:.3f}")

In [7]:
from PIL import Image
import torch
import torch.nn.functional as F
import numpy as np


def pil_to_tensor_01(pil_img):
    """PIL -> float tensor [3,H,W] in [0,1]"""
    x = np.array(pil_img).astype(np.float32) / 255.0
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)
    if x.shape[-1] == 4:
        x = x[..., :3]
    x = torch.from_numpy(x).permute(2, 0, 1).contiguous()
    return x


def blur_tensor(x_01: torch.Tensor, k: int = 21):
    """
    x_01: [1,3,H,W] in [0,1]
    blur with avg pool; fast & dependency-free
    """
    pad = k // 2
    x = F.pad(x_01, (pad, pad, pad, pad), mode="reflect")
    x = F.avg_pool2d(x, kernel_size=k, stride=1)
    return x

In [ ]:
import torch
import random
from diffusers import DDPMScheduler

os.makedirs("/home/diaoyueqin/hcy/Generation/CA", exist_ok=True)

generator = Generator4Embeds(num_inference_steps=4, device="cuda", enable_control_adapter=True)
generator.control_adapter.to("cuda", dtype=torch.float32)
enable_only_control_adapter_trainable(generator.pipe, generator.control_adapter)

# (A) 推理用 scheduler：保持原样（你打印过是 EulerA）
infer_scheduler = generator.pipe.scheduler

# (B) 训练用 scheduler：DDPM（与 randint(0..999) 完全匹配）
train_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    prediction_type="epsilon",
)

print("[init] infer_scheduler =", type(infer_scheduler))
print("[init] train_scheduler =", type(train_scheduler))

optimizer = torch.optim.AdamW(
    (p for p in generator.control_adapter.parameters() if p.requires_grad),
    lr=2e-5, weight_decay=1e-2
)

generator.control_adapter.train()

train = True
texts_train, images_train = load_data()
print("train =", train, "len(images_train) =", len(images_train), "emb_eeg =", emb_eeg.shape[0])

scaler = torch.cuda.amp.GradScaler()
warmup_steps = 2000
base_scale = 0.01
target_scale = 0.1

global_step = 0
EPOCHS = 3

# ====== NEW: 每张图只选 1 个 EEG ======
assert emb_eeg.shape[0] >= len(images_train) * 4, \
    f"Expected emb_eeg >= 4*len(images_train), got {emb_eeg.shape[0]} vs {len(images_train) * 4}"

num_imgs = len(images_train)  # 现在每个 epoch 的 step 数 = num_imgs = 16540
print("steps_per_epoch =", num_imgs, "(1 EEG per image)")

for epoch in range(EPOCHS):
    running = 0.0
    for img_idx in range(num_imgs):
        # ---- 关键：对同一张图的 4 个 EEG，随机选 1 个 ----
        k0 = img_idx * 4
        k = k0 + random.randint(0, 3)
        eeg_embeds = emb_eeg[k:k + 1].to("cuda").float()
        with torch.no_grad():
            h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=16, guidance_scale=5.0)  # [1,1024]

        # ===== 对 h 做稳定化 =====
        h = h.to("cuda", dtype=torch.float32)
        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)

        # L2 normalize（让分布更像 CLIP image embeds）
        h = F.normalize(h, dim=-1)

        # 可选：再做幅度限制（防止极端值）
        h = h.clamp(-3.0, 3.0)

        anneal_steps = 8000  # 你数据量大就设大些
        p_coarse_max = 0.8  # 最终 80% 用 coarse-control
        coarse_steps = 2  # coarse 阶段少步数就行（省显存/时间）

        # ---- blur GT image -> blurry_gt ----
        gt_path = images_train[img_idx]
        gt_pil = Image.open(gt_path).convert("RGB").resize((1024, 1024), Image.BICUBIC)
        gt_01 = pil_to_tensor_01(gt_pil).unsqueeze(0).to("cuda", dtype=torch.float16)
        blurry_gt = blur_tensor(gt_01, k=21)

        # ---- p_coarse 退火 ----
        frac2 = min(1.0, global_step / anneal_steps)
        p_coarse = p_coarse_max * frac2

        # ---- 生成 coarse-control（不进图，不反传）----
        use_coarse = (random.random() < p_coarse)
        # use_coarse = True
        if use_coarse:
            with torch.no_grad():
                # 用同一个 generator 先出 coarse（不启用 control）
                # 这里可以暂时把 generator.num_inference_steps 改成更小
                # 关键：推理前切回 Euler（coarse 会 set_timesteps(2)）
                generator.pipe.scheduler = infer_scheduler
                old_steps = generator.num_inference_steps
                generator.num_inference_steps = coarse_steps

                coarse_pil = generator.generate(
                    h.to(dtype=torch.float16),
                    control_image=None,
                    enable_control=False,
                    control_scale=0.0,
                )

                generator.num_inference_steps = old_steps

            coarse_01 = pil_to_tensor_01(coarse_pil).unsqueeze(0).to("cuda", dtype=torch.float16)
            blurry_tensor = blur_tensor(coarse_01, k=21)
        else:
            blurry_tensor = blurry_gt

        # ===== warmup control_scale =====
        frac = min(1.0, global_step / warmup_steps)
        scale = base_scale + (target_scale - base_scale) * frac

        optimizer.zero_grad(set_to_none=True)
        generator.pipe.scheduler = train_scheduler

        with torch.amp.autocast('cuda', dtype=torch.float16):
            loss = generator.pipe.train_step_control_adapter(
                target_images=gt_01,
                ip_adapter_embeds=h.to(torch.float16),
                control_image=blurry_tensor,
                prompt="",
                control_adapter=generator.control_adapter,
                control_scale=scale,
            )

        # optimizer.zero_grad(set_to_none=True)
        # loss.backward()
        # optimizer.step()
        # 
        # running += loss.item()

        # 断言：防止 loss 断图
        if not loss.requires_grad:
            raise RuntimeError("loss is detached: ControlAdapter not in graph (check residual injection).")

        scaler.scale(loss).backward()

        # 可选：梯度裁剪（建议加）
        # scaler.unscale_(optimizer)
        # torch.nn.utils.clip_grad_norm_(generator.control_adapter.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item())
        global_step += 1
        if (img_idx + 1) % 20 == 0:
            print(
                f"epoch {epoch} step {img_idx + 1}/{num_imgs} loss {running / 20:.4f} scale={scale:.4g} p_coarse={p_coarse:.3f}")
            running = 0.0

    torch.save(generator.control_adapter.state_dict(),
               f"/home/diaoyueqin/hcy/Generation/CA/ckpt_newmamto_control_adapter_epoch{epoch}.pt")

    print("saved adapter.")

In [ ]:

# Assuming generator.generate returns a PIL Image
from PIL import Image
import os
train = False
texts_test, images_test = load_data()
print("train =", train, "len(images_test) =", len(images_test), "emb_eeg_test =", emb_eeg_test.shape[0])

# 重新 new 一个 generator（强烈建议：推理与训练分离）
generator = Generator4Embeds(num_inference_steps=4, device="cuda", enable_control_adapter=True)

adapter_ckpt = "/home/diaoyueqin/hcy/Generation/CA/ckpt_newmamto_control_adapter_epoch0.pt"
assert os.path.exists(adapter_ckpt), f"ckpt not found: {adapter_ckpt}"

generator.control_adapter.load_state_dict(torch.load(adapter_ckpt, map_location="cuda"))
generator.control_adapter.to("cuda", dtype=torch.float32)
generator.control_adapter.eval()
print("Loaded control adapter:", adapter_ckpt)
    
directory = f"/home/diaoyueqin/hcy/Generation/RF_CA_mamto_generated_imgs/{sub}"
with torch.no_grad():
    for k in range(200):
        eeg_embeds = emb_eeg_test[k:k+1]
        h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=16, guidance_scale=5.0)
        # ===== 对 h 做稳定化 =====
        h = h.to("cuda", dtype=torch.float32)
        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
    
        # L2 normalize（让分布更像 CLIP image embeds）
        h = F.normalize(h, dim=-1)
    
        # 可选：再做幅度限制（防止极端值）
        h = h.clamp(-3.0, 3.0)
        
        # ===== Stage-A: coarse（不启用 control）=====
        coarse = generator.generate(
            h.to(dtype=torch.float16),
            control_image=None,
            enable_control=False,
            control_scale=0.0,
        )  # PIL
    
        # coarse -> tensor -> blur 作为 control
        coarse_01 = pil_to_tensor_01(coarse).unsqueeze(0).to("cuda", dtype=torch.float32)
        control_tensor = blur_tensor(coarse_01, k=21)
        
        # ===== Stage-B: refine（启用 control）=====
        for j in range(10):
            image = generator.generate(
                h.to(dtype=torch.float16),
                control_image=control_tensor,
                enable_control=True,
                control_scale=0.01
            )
            path = f'{directory}/{texts_test[k]}/{j}.png'
            os.makedirs(os.path.dirname(path), exist_ok=True)
            image.save(path)
            print(f'Image saved to {path}')


In [ ]:
# from PIL import Image
# import os
# 
# # Assuming generator.generate returns a PIL Image
# generator = Generator4Embeds(num_inference_steps=4, device=device)
# 
# directory = f"/home/diaoyueqin/hcy/Generation/RF_CA_mamto_generated_imgs/{sub}"
# for k in range(200):
#     eeg_embeds = emb_eeg_test[k:k+1]
#     h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=16, guidance_scale=5.0)
#     for j in range(10):
#         image = generator.generate(h.to(dtype=torch.float16))
#         # Construct the save path for each image
#         path = f'{directory}/{texts[k]}/{j}.png'
#         # Ensure the directory exists
#         os.makedirs(os.path.dirname(path), exist_ok=True)
#         # Save the PIL Image
#         image.save(path)
#         print(f'Image saved to {path}')